#### Varying pos:neg batch composition experiment. 

In [1]:
from pathlib import Path

import numpy as np
import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
from pytorch_lightning.loggers import CSVLogger

import arrays as array_io
import datasets
import models
import reporting
from features import FEATURE_COLS
from folds import assign_folds, log_fold_stats
from population_stats import POPULATION_STATS_PATH, load_population_stats

ARRAY_DIR = array_io.ARRAY_DIR
RESULTS_DIR = Path("results/mlp_pos_neg_batch_experiment")
BATCH_SIZE = 2048
N_FOLDS = 3
RECALL_TARGET = 0.80  # fixed recall point for the precision-at-recall sweep

# Target pos:neg ratios, plus the natural rate read off population_stats.json
# (165,908 positive of 59.1M total, about 1:356) instead of hardcoding an
# approximate 1:350 -- if the array gets rebuilt off different source parquet
# this stays correct without editing by hand.
POS_NEG_RATIOS = [1, 5, 10, 50]
_population_stats = load_population_stats(POPULATION_STATS_PATH)
NATURAL_RATIO = _population_stats["total_negative"] / _population_stats["total_positive"]
POS_NEG_RATIOS.append(NATURAL_RATIO)
POS_PER_BATCH = [max(1, round(BATCH_SIZE / (ratio + 1))) for ratio in POS_NEG_RATIOS]


In [2]:
def run_fold(fold_idx, data_module, fold_dir):
    index = data_module.index
    model = models.MultiLayerPerceptron(len(FEATURE_COLS))

    fold_dir.mkdir(parents=True, exist_ok=True)
    fold_name = f"fold_{fold_idx}"
    logger = CSVLogger(save_dir=str(fold_dir.parent), name=fold_name)
    checkpoint = ModelCheckpoint(monitor="val_loss", mode="min", save_top_k=1)
    trainer = pl.Trainer(
        max_epochs = 15,
        callbacks= [checkpoint],
        logger=logger,
        enable_progress_bar=True,
        accelerator="auto"
    )

    trainer.fit(model, datamodule=data_module)
    epochs_run = trainer.current_epoch
    best_model = models.MultiLayerPerceptron.load_from_checkpoint(checkpoint.best_model_path)

    probs, labels = reporting.get_val_predictions(best_model, data_module.val_dataloader())
    np.savez(fold_dir / "val_preds.npz", probs=probs, labels=labels)
    train_probs, train_labels = reporting.get_val_predictions(
        best_model, data_module.train_eval_dataloader()
    )
    np.savez(fold_dir / "train_preds.npz", probs=train_probs, labels=train_labels)

    metrics_csvs = sorted(fold_dir.glob("**/metrics.csv"))
    reporting.plot_loss_curve(
        metrics_csvs[-1], fold_dir / "loss_curve.png", f"fold {fold_idx} loss"
    )
    reporting.plot_pr_curve_train_val(
        train_labels, train_probs, labels, probs, out_path=fold_dir / "pr_curve.png",
        title=f"fold {fold_idx}: train vs validation precision-recall curve",
    )

    fold_score = reporting.score_fold(labels, probs)
    train_score = reporting.score_fold(train_labels, train_probs)
    fold_score.update({f"train_{k}": v for k, v in train_score.items()})
    fold_score.update({
        "fold": fold_idx,
        "epochs_run": epochs_run,
        "n_train_blocks": index.n_train_blocks,
        "n_val_blocks": index.n_val_blocks,
        "n_train_rows": index.n_train_rows,
        "n_val_rows": index.n_val_rows,
        "train_positives": int(index.train_pos.size),
        "batch_pos_weight_available": data_module.batch_pos_weight,
        "dist_mean_m": float(data_module.dist_mean),
        "dist_std_m": float(data_module.dist_std),
    })
    return fold_score


In [ ]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
arrays, manifest = array_io.load_arrays(ARRAY_DIR)
array_io.populate(arrays)

population_stats = load_population_stats(POPULATION_STATS_PATH)
fold_assignment = assign_folds(population_stats["block_row_counts"], N_FOLDS, 0)
fold_code = array_io.fold_of_code(manifest, fold_assignment)

ratio_summary_rows = []   # one row per (ratio, fold): auprc, precision_at_recall, etc.
ratio_fold_preds = {}     # realized ratio -> list of (train_labels, train_probs, val_labels, val_probs)

for ratio_idx, pos_per_batch in enumerate(POS_PER_BATCH):
    ratio_dir = RESULTS_DIR / str(ratio_idx)
    # Realized ratio, not the POS_NEG_RATIOS target -- rounding pos_per_batch
    # to an integer shifts it slightly, most at the sparse end of the sweep.
    ratio = (BATCH_SIZE - pos_per_batch) / pos_per_batch

    all_fold_rows = []
    for fold_idx in range(N_FOLDS):
        data_module = datasets.ArrayDataModule(arrays, manifest, fold_code, fold_idx,
                    batch_size=BATCH_SIZE, pos_per_batch=pos_per_batch, steps_per_epoch=None)
        data_module.setup()

        fold_score = run_fold(fold_idx, data_module, ratio_dir / f"fold_{fold_idx}")
        fold_score["ratio"] = ratio
        fold_score["pos_per_batch"] = pos_per_batch
        all_fold_rows.append(fold_score)

    fold_dirs = [ratio_dir / f"fold_{fold_idx}" for fold_idx in range(N_FOLDS)]
    metrics_csvs = [c for d in fold_dirs for c in sorted(d.glob("**/metrics.csv"))]
    reporting.plot_mean_loss_curve(
        metrics_csvs,
        ratio_dir / "loss_curve_mean.png",
        f"Mean training and validation loss, batch ratio 1:{ratio:.0f}",
        ylabel="focal loss",
    )

    fold_preds = []
    for d in fold_dirs:
        train_npz = np.load(d / "train_preds.npz")
        val_npz = np.load(d / "val_preds.npz")
        fold_preds.append(
            (train_npz["labels"], train_npz["probs"], val_npz["labels"], val_npz["probs"])
        )

    reporting.plot_mean_pr_curve(
        fold_preds,
        ratio_dir / "pr_curve_mean.png",
        f"Mean precision recall curve, batch ratio 1:{ratio:.0f}",
    )

    for fold_score, (_, _, val_labels, val_probs) in zip(all_fold_rows, fold_preds):
        fold_score["precision_at_recall"] = reporting.precision_at_recall(
            val_labels, val_probs, RECALL_TARGET
        )

    ratio_summary_rows.extend(all_fold_rows)
    ratio_fold_preds[ratio] = fold_preds

reporting.write_ratio_summary(ratio_summary_rows, RESULTS_DIR / "ratio_summary.csv")

# a) AUPRC vs. ratio, fold mean +/- std
reporting.plot_auprc_vs_ratio(
    ratio_summary_rows,
    RESULTS_DIR / "auprc_vs_ratio.png",
    title="Validation AUPRC vs. training batch pos:neg ratio",
)

# b) overlaid PR curves, one line per ratio
reporting.plot_pr_curves_by_ratio(
    ratio_fold_preds,
    RESULTS_DIR / "pr_curves_by_ratio.png",
    title="Validation precision-recall curve by training batch ratio",
)

# c) precision at a fixed recall target vs. ratio
reporting.plot_precision_at_recall_vs_ratio(
    ratio_summary_rows,
    RECALL_TARGET,
    RESULTS_DIR / "precision_at_recall_vs_ratio.png",
    title=f"Validation precision at {RECALL_TARGET:.0%} recall vs. training batch ratio",
)



populate emb: 22.76 GB in 655.9s (0.03 GB/s)


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name            | Type            | Params | Mode  | FLOPs
--------------------------------------------------------------------
0 | model           | Sequential      | 66.2 K | train | 0    
1 | loss_fn         | BinaryFocalLoss | 0      | train | 0    
2 | train_precision | BinaryPrecision | 0      | train | 0    
3 | train_recall    | BinaryRecall    | 0      | train | 0    
4 | train_f1        | BinaryF1Score   | 0      | train | 0    
5 | val_precision   | BinaryPrecision | 0      | train | 0    
6 | val_recall      | BinaryRecall    | 0      | train | 0    
7 | val_f1          | BinaryF1Score   | 0      | train | 0    
-----------

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/druckenmillerlab/miniconda3/envs/wetlands/lib/python3.11/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/druckenmillerlab/miniconda3/envs/wetlands/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


/home/druckenmillerlab/miniconda3/envs/wetlands/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


...

`Trainer.fit` stopped: `max_epochs=15` reached.


Epoch 14: 100%|██████████| 2708/2708 [00:50<00:00, 54.01it/s, v_num=0, val_loss=0.00502]


/home/druckenmillerlab/wetland_conversion/reporting.py:104: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  fig.savefig(out_path)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name            | Type            | Params | Mode  | FLOPs
--------------------------------------------------------------------
0 | model           | Sequential      | 66.2 K | train | 0    
1 | loss_fn         | BinaryFocalLoss | 0      | train | 0    
2 | train_precision | BinaryPrecision | 0      | train | 0    
3 | train_recall    | BinaryRecall    | 0      | train | 0    
4 | train_f1        | BinaryF1Score   | 0      | train | 0    
5 | val_precision   | BinaryPre

/home/druckenmillerlab/miniconda3/envs/wetlands/lib/python3.11/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
/home/druckenmillerlab/miniconda3/envs/wetlands/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.
/home/druckenmillerlab/miniconda3/envs/wetlands/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


Epoch 2:  90%|████████▉ | 16763/18672 [02:46<00:18, 100.69it/s, v_num=0, val_loss=0.00357]